# Forecasting

With the [time-series fundamentals](time-series-fundamentals.ipynb) in place —
chronological splitting, lag features — we can forecast. Good practice first:
establish a **baseline** so a "real" model has something concrete to beat.

In [ ]:
// Same synthetic series as the fundamentals chapter (trend + weekly season).
let series: Vec<f64> = (0..56).map(|t| {
    10.0 + 0.2 * t as f64 + 3.0 * ((t as f64) * 2.0 * std::f64::consts::PI / 7.0).sin()
}).collect();
let period = 7usize;
let split = 44usize;
let horizon = series.len() - split;   // forecast the final 12 days

// Seasonal-naive baseline: predict each day using the value one week earlier.
let seasonal_naive: Vec<f64> = (0..horizon)
    .map(|i| series[split - period + (i % period)])
    .collect();
println!("horizon = {} days", horizon);
println!("seasonal-naive forecast starts: {:.2}, {:.2}, {:.2} ...",
         seasonal_naive[0], seasonal_naive[1], seasonal_naive[2]);

## An MSTL model

[`augurs`](https://docs.rs/augurs) provides MSTL (Multiple Seasonal-Trend
decomposition), which models the trend and seasonal structure and forecasts
forward. We fit on the training window and predict the horizon:

In [ ]:
:dep augurs = { version = "0.10", features = ["mstl", "ets"] }
use augurs::mstl::MSTLModel;
use augurs::prelude::*;

// Fitted-model type isn't nameable across cells, so we return the forecast Vec.
let mstl_forecast: Vec<f64> = {
    let train = &series[..split];
    let model = MSTLModel::naive(vec![period]);
    let fitted = model.fit(train).unwrap();
    fitted.predict(horizon, None).unwrap().point
};
println!("MSTL forecast starts: {:.2}, {:.2}, {:.2} ...",
         mstl_forecast[0], mstl_forecast[1], mstl_forecast[2]);

## Evaluation

Score both forecasts against the held-out actuals with the standard time-series
metrics — **MAE**, **RMSE**, and **MAPE** — computed over the chronological test
window. The model earns its keep only if it beats the baseline:

In [ ]:
{
    let actual = &series[split..];
    let report = |name: &str, pred: &[f64]| {
        let n = actual.len() as f64;
        let mae = actual.iter().zip(pred).map(|(a, p)| (a - p).abs()).sum::<f64>() / n;
        let rmse = (actual.iter().zip(pred).map(|(a, p)| (a - p).powi(2)).sum::<f64>() / n).sqrt();
        let mape = actual.iter().zip(pred).map(|(a, p)| ((a - p) / a).abs()).sum::<f64>() / n * 100.0;
        println!("{:<15} MAE={:.3}  RMSE={:.3}  MAPE={:.1}%", name, mae, rmse, mape);
    };
    report("seasonal-naive", &seasonal_naive);
    report("MSTL", &mstl_forecast);
}

```{note}
**Ecosystem maturity.** `augurs` is a real, actively maintained time-series
toolkit (forecasting, seasonality/changepoint/outlier detection). But Rust's
time-series ecosystem is younger and narrower than Python's (`statsmodels`,
`prophet`, `sktime`) — verify `augurs`' current API against its docs before
relying on it, since it's actively developed.
```

Next: [Working with larger-than-memory data](../11-larger-than-memory/streaming-and-lazy-execution.ipynb) —
scaling the data pipeline beyond what fits in RAM.